In [ ]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import os
from dotenv import load_dotenv
from git import Repo

In [2]:
repo_path = './test_repo'

repo = Repo.clone_from("https://github.com/langchain-ai/langchain", to_path=repo_path)


In [4]:
loader = GenericLoader.from_filesystem(
    repo_path + "/libs/core/langchain_core/",
    glob="**/*", # Pega tudo que tem nessa pasta mais as subpastas
    suffixes = [".py"],
    exclude=["**/non-utf-8-encoding.py"],
    parser= LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()
len(documents)

599

In [ ]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=2000,
    chunk_overlap=200,
)

texts = python_splitter.split_documents(documents)
len(texts)

1680

In [12]:
load_dotenv()

True

In [14]:
embeddings_model = OpenAIEmbeddings(disallowed_special=())
db = Chroma.from_documents(
    texts,
    embedding=embeddings_model,
)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k":8}
)


In [19]:
llm = ChatOpenAI(model="gpt-3.5-turbo",max_tokens=1000)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um revisor de código experiente. Forneca informações detalhadas sobre a revisão do código e sugestões de melhorias baseado no contexto fornecido abaixo: \n\n {context}"
        ),
        (
            "user",
            "{input}"
        )
    ]
)

document_chain = create_stuff_documents_chain(
    llm,
    prompt    
)

retrieval_chain = create_retrieval_chain(retriever,document_chain)

In [20]:
response = retrieval_chain.invoke({"input": "Você pode revisar e sugerir melhorias para o codigo RunnableBinding?"})
print(response['answer'])

### Revisão e Sugestões de Melhorias para o Código `RunnableBinding`:

1. **Documentação:**
    - A documentação presente na `bind` e no método `with_config` é clara e informativa, o que é ótimo. Incluir documentação detalhada é essencial para entender a finalidade e uso dos métodos.

2. **Assinatura de Método:**
    - A assinatura dos métodos `bind` e `with_config` está correta e mostra uma boa prática de programação.

3. **Tipagem:**
    - A tipagem está bem definida para os parâmetros e retorno dos métodos, o que é altamente recomendado para garantir a segurança e a clareza do código.

4. **Tratamento de Configuração:**
    - No método `with_config`, a forma como as configurações são mescladas é boa, mas pode ser melhorada em termos de clareza, separando a lógica de mesclagem em um método adicional para facilitar a manutenção e compreensão.

5. **Unificação de Tratamento de Argumentos:**
    - Seria interessante unificar a lógica para lidar com os argumentos adicionais em um método 